# HURDLER success rate, 1–60AA (figure through 50AA)

Uses three repeated copies to complete every cyclic 3-mer window while retaining the original per-plasmid pattern population, distance criterion, random inputs, and get_re_sites.ipynb typography, lines, text, and plasmid order on a near-square 6×5-inch canvas. The historical two-copy result remains the exact comparison baseline; the authoritative plot displays 1–50AA.

**Rules:** `historical-notebook-success-v1-three-copy-scan`; **seed:** 42 unless explicitly noted.

In [ ]:
REPO = '/home/wendai/projects/hurdler/clone_repeat_protein'
RULE_PROFILE = 'historical-notebook-success-v1-three-copy-scan'
TWO_SHORT_RESULTS = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/runs/run03_single_file_16core/raw/short_motifs_1_5.parquet'
TWO_RANDOM_RESULTS = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/runs/run03_single_file_16core/raw/random_modules_6_60.parquet'
SHORT_RESULTS = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/runs/run06_three_copy_16core/raw/short_motifs_1_5.parquet'
RANDOM_RESULTS = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/runs/run06_three_copy_16core/raw/random_modules_6_60.parquet'
COMPARISON = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/tables/scan_copy_comparison_2x_vs_3x.csv'
BY_LENGTH = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/tables/scan_copy_improvement_by_length.csv'
LANDSCAPE_SCRIPT = '/home/wendai/projects/hurdler/clone_repeat_protein/scripts/run_success_landscape_single_files.py'
FIGURE_DIR = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step02_success_landscape/figures/scan_3x'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd

def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

run_context = {'rule_profile': RULE_PROFILE, 'input_hashes': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': []}

In [ ]:
import importlib.util
import pyarrow.parquet as pq
spec = importlib.util.spec_from_file_location('single_file_success_landscape', LANDSCAPE_SCRIPT)
landscape = importlib.util.module_from_spec(spec)
spec.loader.exec_module(landscape)
paths = {'two_short': TWO_SHORT_RESULTS, 'two_random': TWO_RANDOM_RESULTS, 'three_short': SHORT_RESULTS, 'three_random': RANDOM_RESULTS, 'comparison': COMPARISON, 'by_length': BY_LENGTH}
run_context['input_hashes'] = {name: sha256(path) for name, path in paths.items()}
rates = landscape.success_summary(Path(SHORT_RESULTS), Path(RANDOM_RESULTS))
two_copy_rates = landscape.success_summary(Path(TWO_SHORT_RESULTS), Path(TWO_RANDOM_RESULTS))
comparison = pd.read_csv(COMPARISON)
by_length = pd.read_csv(BY_LENGTH)
short_rows = pq.ParquetFile(SHORT_RESULTS).metadata.num_rows
random_rows = pq.ParquetFile(RANDOM_RESULTS).metadata.num_rows
assert short_rows == 3_368_420
assert random_rows == 440_000
assert len(rates) == len(two_copy_rates) == len(comparison) == 60 * 8
assert not {'ci95_low', 'ci95_high'}.intersection(rates.columns)
assert comparison.success_delta.ge(0).all()
assert rates.loc[rates.module_length.le(2), 'successes'].sum() == 0
improved_lengths = by_length.loc[by_length.any_improvement, 'module_length'].astype(int).tolist()
run_context['row_counts'] = {'rate_rows': len(rates), 'short_motifs': short_rows, 'sampled_sequences': random_rows, 'improved_length_plasmid_rows': int(comparison.improved.sum()), 'improved_module_lengths': len(improved_lengths)}
run_context['filter_flow'] = ['rebuild the original notebook plasmid-specific pattern population', 'keep identical exhaustive and random module inputs for 2x and 3x', 'repeat short motif to shortest >=6AA module', 'scan module+module+module with 5 <= d < effective module length', 'compare every length/plasmid to the historical two-copy baseline']
run_context['limitations'] = ['6-60AA values are seeded Monte Carlo point estimates; uncertainty intervals are intentionally not drawn']
rates.head()

In [ ]:
by_length[['module_length','improved_plasmids','added_successes_all_plasmids','mean_success_rate_2x','mean_success_rate_3x','maximum_plasmid_rate_delta','improved_plasmid_names']]

In [ ]:
from IPython.display import Image, display
figures = landscape.plot_success_curve(rates, Path(FIGURE_DIR), file_stem='success_rate_1_60_scan_3x')
display(Image(filename=str(Path(FIGURE_DIR) / 'success_rate_1_60_scan_3x.png')))
rates.groupby(['method','plasmid']).agg(lengths=('module_length','nunique'), tests=('tests','sum'), successes=('successes','sum')).reset_index()

Every sequence is now scanned as `module + module + module`; the distance upper bound remains the single effective module length. The third copy completes the cyclic window whose Site-I 3-mer begins at the final residue of the first copy and whose Site-II start is `L-1` residues later. It adds no new biological motif or pattern rule. The 1AA and 2AA exhaustive rates remain zero for all eight plasmids. The plot uses a near-square 6×5-inch canvas with the original notebook's default font and line settings, plasmid order, exact title `3-mer Probability vs Sequence Length`, and 50AA upper limit. It shows one point estimate per length and plasmid, with no confidence band or method-divider annotation.